# 03 — Finalize
Trim to the published variable set, fix dtypes and known metadata issues,
clean global attributes, and align to CF-1.11 before saving the final,
publication-ready dataset.

In [ ]:
import sys
import yaml
import xarray as xr
sys.path.insert(0, '..')

from nitrate import cf, qc

In [ ]:
config = yaml.safe_load(open('../config/GI01SUMO-SBD11-08-NUTNRB000.yaml'))
refdes = config['refdes']
data_dir = f"../{config['paths']['data_dir']}"

data = xr.open_dataset(f'{data_dir}{refdes}_bottle_corrected.nc').load()
data

## 1. Trim and rename to the published variable set

In [ ]:
save_vars = config['save_vars']

final = data[save_vars]
final = final.rename({
    'corrected_nitrate_concentration_mad': 'burst_median_absolute_deviation',
    'nitrate_sensor_quality_flag': 'nitrate_concentration_qc_flag',
})
final['burst_median_absolute_deviation'].attrs['comment'] = (
    'The median absolute deviation calculated for each sampling burst.')
final

## 2. Fix dtypes
A few variables may have drifted to float during resampling/concatenation.

In [ ]:
final['drift_corrected_nitrate_qc_flag'] = final['drift_corrected_nitrate_qc_flag'].astype('int')
final['nitrate_concentration_qc_flag'] = final['nitrate_concentration_qc_flag'].astype('int')
final['deployment'] = final['deployment'].astype('int')
final['serial_number'] = final['serial_number'].astype('int')

## 3. Patch known metadata issues
Applies any (deployment, serial_number) corrections listed in the config.

In [ ]:
for depNum, serial_number in config.get('serial_number_fixes', []):
    final = cf.fix_serial_numbers(final, depNum, serial_number)

## 4. Clean global attributes and flag not-evaluated records

In [ ]:
final = cf.clean_netcdf(final)
final = qc.add_not_evaluated_flags(final, 'bottle_corrected_nitrate')
final = qc.add_not_evaluated_flags(final, 'drift_corrected_nitrate')

## 5. CF-1.11 alignment

In [ ]:
final = cf.finalize_cf_compliance(final)
final

## 6. Save the final dataset

In [ ]:
outpath = f'{data_dir}{refdes}.nc'
final.to_netcdf(outpath, format='netcdf4', engine='h5netcdf')
print(f'Saved -> {outpath}')